# Analisis Jaringan Kolaborasi Dosen Informatika

Notebook ini melakukan **Exploratory Network Analysis** lengkap terhadap dataset publikasi ilmiah dosen Departemen Informatika/Ilmu Komputer. Pipeline dibagi menjadi lima fase:

1. **Data Loading & Audit** muat semua dataset, cek kondisi awal
2. **Preprocessing** bersihkan, normalisasi, dan integrasi data
3. **Exploratory Data Analysis (EDA)** statistik deskriptif dan tren publikasi
4. **Network Analysis** bangun graf kolaborasi dan hitung metrik jaringan
5. **Import ke Neo4j** muat data ke graph database untuk query lanjutan

## 0. Install & Import Library

Semua library yang dibutuhkan diinstall dan diimport di sini. Jalankan sekali di awal.

In [ ]:
import subprocess
import sys

packages = ['pandas', 'numpy', 'matplotlib', 'seaborn', 'networkx', 'neo4j', 'tqdm']
for pkg in packages:
    subprocess.check_call([sys.executable, '-m', 'pip', 'install', pkg, '-q'])

print('Semua library berhasil diinstall.')

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import networkx as nx
import warnings
import re
from collections import Counter
from itertools import combinations

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 80)
pd.set_option('display.float_format', '{:.2f}'.format)

plt.rcParams.update({
    'figure.dpi': 120,
    'font.family': 'DejaVu Sans',
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3,
})

PALETTE = ['#2E4057', '#048A81', '#54C6EB', '#EF946C', '#C4A77D']

print('Import selesai. Siap analisis.')

Import selesai. Siap analisis.


## FASE 1 (Data Loading & Audit)

Muat ketiga file CSV dan lakukan audit awal: jumlah baris, kolom, tipe data, dan missing values. Tujuannya memahami kondisi data sebelum diproses.

In [11]:
PATH_DOSEN   = 'dosen_infokom_final.csv'
PATH_SCHOLAR = 'dosen_papers_scholar.csv'
PATH_SCOPUS  = 'dosen_papers_scopus.csv'

dosen   = pd.read_csv(PATH_DOSEN)
scholar = pd.read_csv(PATH_SCHOLAR)
scopus  = pd.read_csv(PATH_SCOPUS)

print('RINGKASAN DATASET')
print(f'  dosen_infokom_final  : {dosen.shape[0]:>5} baris  x {dosen.shape[1]} kolom')
print(f'  dosen_papers_scholar : {scholar.shape[0]:>5} baris  x {scholar.shape[1]} kolom')
print(f'  dosen_papers_scopus  : {scopus.shape[0]:>5} baris  x {scopus.shape[1]} kolom')
print(f'  Total publikasi      : {scholar.shape[0] + scopus.shape[0]:>5} paper')

RINGKASAN DATASET
  dosen_infokom_final  :   129 baris  x 9 kolom
  dosen_papers_scholar :  5573 baris  x 15 kolom
  dosen_papers_scopus  :  1151 baris  x 11 kolom
  Total publikasi      :  6724 paper


Cek detail kolom dan tipe data masing-masing file.

In [3]:
for label, df in [('DOSEN', dosen), ('SCHOLAR', scholar), ('SCOPUS', scopus)]:
    print(f'--- {label} ---')
    print(df.dtypes.to_string())
    print()

--- DOSEN ---
nama_dosen     object
nama_norm      object
nip           float64
nidn          float64
prodi          object
scholar_id     object
scopus_id      object
sinta_id      float64
source         object

--- SCHOLAR ---
Authors           object
Author IDs        object
Title             object
Year             float64
Journal           object
Link              object
Abstract         float64
Keywords         float64
Document Type    float64
DOI              float64
TLDR             float64
citation_id       object
scholar_id        object
dosen             object
source            object

--- SCOPUS ---
Authors          object
Author IDs       object
Title            object
Year              int64
Journal          object
Link             object
Abstract         object
Keywords         object
Document Type    object
DOI              object
TLDR             object



Audit missing values secara lengkap untuk setiap dataset.

In [12]:
def audit_missing(df, label):
    total = len(df)
    missing = df.isnull().sum()
    pct = (missing / total * 100).round(1)
    report = pd.DataFrame({'Missing': missing, 'Persen (%)': pct})
    report = report[report['Missing'] > 0].sort_values('Missing', ascending=False)
    print(f'Missing Values: {label}')
    if report.empty:
        print('  Tidak ada missing values.')
    else:
        print(report.to_string())
    print()

audit_missing(dosen, 'dosen_infokom_final')
audit_missing(scholar, 'dosen_papers_scholar')
audit_missing(scopus, 'dosen_papers_scopus')

Missing Values: dosen_infokom_final
          Missing  Persen (%)
sinta_id       22       17.10
nidn           14       10.90

Missing Values: dosen_papers_scholar
               Missing  Persen (%)
Abstract          5573      100.00
Keywords          5573      100.00
Document Type     5573      100.00
DOI               5573      100.00
TLDR              5573      100.00
Journal            232        4.20
Year               227        4.10

Missing Values: dosen_papers_scopus
          Missing  Persen (%)
TLDR          776       67.40
Keywords      272       23.60
DOI            66        5.70
Abstract        7        0.60



Cek duplikat, paper yang muncul lebih dari sekali dalam dataset yang sama.

In [5]:
dup_scholar = scholar.duplicated(subset='citation_id').sum()
dup_scopus  = scopus.duplicated(subset='DOI').sum()
dup_dosen   = dosen.duplicated(subset='nidn').sum()

print(f'Duplikat di scholar (citation_id) : {dup_scholar}')
print(f'Duplikat di scopus  (DOI)         : {dup_scopus}')
print(f'Duplikat di dosen   (nidn)        : {dup_dosen}')

Duplikat di scholar (citation_id) : 0
Duplikat di scopus  (DOI)         : 65
Duplikat di dosen   (nidn)        : 13


## FASE 2 (Preprocessing)

Fase ini membersihkan dan menyiapkan data agar siap dianalisis. Ada empat langkah utama:
- Normalisasi nama dosen
- Bersihkan kolom tahun
- Ekstrak nama jurnal dari string panjang
- Bangun tabel relasi dosen-paper

### 2.1 Normalisasi Nama Dosen

Nama dosen di kolom `Authors` sering tidak konsisten. Kita buat fungsi normalisasi dan daftar lookup dari data dosen.

In [6]:
def normalize_name(name):
    if pd.isna(name):
        return ''
    name = str(name).strip()
    name = re.sub(r'\s+', ' ', name)
    prefixes = r'^(Dr\.|Prof\.|Ir\.|Drs\.|Dra\.|S\.T\.|S\.Kom\.|M\.Kom\.|M\.T\.|Ph\.D\.|S\.Pd\.|M\.Pd\.|M\.Si\.|S\.Si\.|S\.Pd\.T\.|\.T\.)'
    name = re.sub(prefixes, '', name, flags=re.IGNORECASE).strip()
    name = re.sub(r',.*', '', name).strip()
    return name.title()

dosen['nama_norm_clean'] = dosen['nama_norm'].apply(normalize_name)

dosen_lookup = dict(zip(dosen['nama_norm_clean'], dosen['nama_norm']))
dosen_set    = set(dosen['nama_norm_clean'])

print(f'Total dosen dalam lookup : {len(dosen_set)}')
print('\nContoh nama normalisasi:')
print(dosen[['nama_dosen', 'nama_norm', 'nama_norm_clean']].head(8).to_string(index=False))

Total dosen dalam lookup : 129

Contoh nama normalisasi:
                             nama_dosen             nama_norm       nama_norm_clean
          Aditya Prapanca, S.T., M.Kom.       Aditya Prapanca       Aditya Prapanca
          Anita Qoiriah, S.Kom., M.Kom.         Anita Qoiriah         Anita Qoiriah
      Dr. Yuni Yamasari, S.Kom., M.Kom.         Yuni Yamasari         Yuni Yamasari
            Agus Prihanto, S.T., M.Kom.         Agus Prihanto         Agus Prihanto
Dr. Ir. Ricky Eka Putra, S.Kom., M.Kom.       Ricky Eka Putra       Ricky Eka Putra
        I Made Suartana, S.Kom., M.Kom.       I Made Suartana       I Made Suartana
          Naim Rochmawati, S.Kom., M.T.       Naim Rochmawati       Naim Rochmawati
   Paramitha Nerisafitra, S.ST., M.Kom. Paramitha Nerisafitra Paramitha Nerisafitra


### 2.2 Preprocessing Dataset Scholar

Bersihkan kolom Year, ekstrak nama jurnal saja (tanpa volume/halaman), dan tandai paper mana yang ditulis oleh dosen Infokom.

In [7]:
scholar_clean = scholar.copy()

scholar_clean['Year'] = pd.to_numeric(scholar_clean['Year'], errors='coerce')
scholar_clean = scholar_clean[scholar_clean['Year'].between(2000, 2026)].copy()
scholar_clean['Year'] = scholar_clean['Year'].astype(int)

def extract_journal_name(journal_str):
    if pd.isna(journal_str):
        return 'Unknown'
    cleaned = re.sub(r'\d+.*$', '', str(journal_str)).strip().rstrip(',')
    return cleaned if cleaned else 'Unknown'

scholar_clean['Journal_clean'] = scholar_clean['Journal'].apply(extract_journal_name)

scholar_clean['source_db'] = 'Scholar'

print(f'Scholar setelah filter tahun valid : {len(scholar_clean)} paper')
print(f'Rentang tahun                       : {scholar_clean["Year"].min()} - {scholar_clean["Year"].max()}')
print(f'\nContoh Journal_clean:')
print(scholar_clean[['Journal', 'Journal_clean']].dropna().head(5).to_string(index=False))

Scholar setelah filter tahun valid : 5336 paper
Rentang tahun                       : 2001 - 2026

Contoh Journal_clean:
                                                                         Journal                                            Journal_clean
             Journal of Informatics and Computer Science (JINACS), 563-567, 2026     Journal of Informatics and Computer Science (JINACS)
               IT-Edu: Jurnal Information Technology and Education 11 (01), 2026      IT-Edu: Jurnal Information Technology and Education
PengabdianMu: Jurnal Ilmiah Pengabdian kepada Masyarakat 10 (8), 1936-1943, 2025 PengabdianMu: Jurnal Ilmiah Pengabdian kepada Masyarakat
        Journal of Informatics and Computer Science (JINACS) 7 (01), 50-56, 2025     Journal of Informatics and Computer Science (JINACS)
    Journal of Informatics and Computer Science (JINACS) 6 (04), 1089-1098, 2025     Journal of Informatics and Computer Science (JINACS)


### 2.3 Preprocessing Dataset Scopus

Scopus memiliki format yang lebih terstruktur. Kita filter tahun, bersihkan jurnal, dan tambahkan flag source.

In [8]:
scopus_clean = scopus.copy()

scopus_clean['Year'] = pd.to_numeric(scopus_clean['Year'], errors='coerce').astype('Int64')
scopus_clean = scopus_clean[scopus_clean['Year'].between(2000, 2026)].copy()

scopus_clean['Journal_clean'] = scopus_clean['Journal'].fillna('Unknown')
scopus_clean['source_db'] = 'Scopus'
scopus_clean['dosen'] = None

print(f'Scopus setelah filter tahun valid   : {len(scopus_clean)} paper')
print(f'Rentang tahun                        : {int(scopus_clean["Year"].min())} - {int(scopus_clean["Year"].max())}')
print(f'Tipe dokumen:')
print(scopus_clean['Document Type'].value_counts().to_string())

Scopus setelah filter tahun valid   : 1151 paper
Rentang tahun                        : 2006 - 2026
Tipe dokumen:
Document Type
Conference paper    641
Article             485
Review                9
Book chapter          8
Editorial             6
Note                  1
Data paper            1


### 2.4 Ekstrak Relasi Dosen - Paper

Kolom `Authors` berisi satu string dengan banyak nama. Kita pecah per nama, lalu cocokkan dengan daftar dosen Infokom. Hasilnya adalah tabel relasi: satu baris = satu dosen terlibat di satu paper.

In [9]:
def parse_authors_scholar(authors_str):
    if pd.isna(authors_str):
        return []
    parts = [a.strip() for a in str(authors_str).split(',')]
    return [normalize_name(p) for p in parts if p]

def parse_authors_scopus(authors_str):
    if pd.isna(authors_str):
        return []
    parts = [a.strip() for a in str(authors_str).split(';')]
    return [normalize_name(p) for p in parts if p]

scholar_clean['author_list'] = scholar_clean['Authors'].apply(parse_authors_scholar)
scopus_clean['author_list']  = scopus_clean['Authors'].apply(parse_authors_scopus)

scholar_clean['infokom_authors'] = scholar_clean['author_list'].apply(
    lambda lst: [a for a in lst if a in dosen_set]
)
scopus_clean['infokom_authors'] = scopus_clean['author_list'].apply(
    lambda lst: [a for a in lst if a in dosen_set]
)

scholar_with_dosen = scholar_clean[scholar_clean['infokom_authors'].map(len) > 0]
scopus_with_dosen  = scopus_clean[scopus_clean['infokom_authors'].map(len) > 0]

print(f'Scholar papers dengan min 1 dosen Infokom : {len(scholar_with_dosen)}')
print(f'Scopus  papers dengan min 1 dosen Infokom : {len(scopus_with_dosen)}')

Scholar papers dengan min 1 dosen Infokom : 5336
Scopus  papers dengan min 1 dosen Infokom : 774


### 2.5 Bangun Tabel Relasi Dosen-Paper (Gabungan)

Explode setiap paper ke baris per dosen, lalu gabungkan Scholar dan Scopus jadi satu tabel relasi terpadu.

In [16]:
scholar_exploded = scholar_with_dosen.explode('infokom_authors')[[
    'citation_id', 'Title', 'Year', 'Journal_clean', 'infokom_authors', 'source_db'
]].rename(columns={'citation_id': 'paper_id', 'infokom_authors': 'nama_norm_clean'})

scopus_clean['aper_id'] = 'scopus_' + scopus_clean.index.astype(str)
scopus_exploded = scopus_with_dosen.explode('infokom_authors')[[
    'paper_id', 'Title', 'Year', 'Journal_clean', 'infokom_authors', 'source_db'
]].rename(columns={'infokom_authors': 'nama_norm_clean'})

relasi = pd.concat([scholar_exploded, scopus_exploded], ignore_index=True)
relasi = relasi.dropna(subset=['nama_norm_clean'])
relasi = relasi[relasi['nama_norm_clean'] != '']

relasi = relasi.merge(dosen[['nama_norm_clean', 'prodi', 'scholar_id', 'scopus_id']], 
                      on='nama_norm_clean', how='left')

print(f'Total relasi dosen-paper : {len(relasi)}')
print(f'Dosen unik terlibat      : {relasi["nama_norm_clean"].nunique()}')
print(f'Paper unik               : {relasi["paper_id"].nunique()}')
print()
print(relasi.head(5).to_string(index=False))

KeyError: "['nama_norm_clean'] not in index"

In [14]:
print(scopus_with_dosen.columns.tolist())

['Authors', 'Author IDs', 'Title', 'Year', 'Journal', 'Link', 'Abstract', 'Keywords', 'Document Type', 'DOI', 'TLDR', 'Journal_clean', 'source_db', 'dosen', 'author_list', 'infokom_authors']


---
## FASE 3 — Exploratory Data Analysis (EDA)

Analisis statistik deskriptif dan visualisasi tren publikasi, produktivitas dosen, distribusi per prodi, dan topik riset.

### 3.1 Statistik Deskriptif Dataset Dosen

In [ ]:
print('=== DISTRIBUSI DOSEN PER PRODI ===')
prodi_dist = dosen['prodi'].value_counts()
print(prodi_dist.to_string())
print(f'\nTotal prodi  : {dosen["prodi"].nunique()}')
print(f'Total dosen  : {len(dosen)}')

Visualisasi distribusi dosen per program studi.

In [ ]:
top_prodi = prodi_dist.head(10)

fig, ax = plt.subplots(figsize=(10, 6))
bars = ax.barh(top_prodi.index[::-1], top_prodi.values[::-1], 
               color=PALETTE[0], edgecolor='white', linewidth=0.5)

for bar, val in zip(bars, top_prodi.values[::-1]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=10, fontweight='bold')

ax.set_xlabel('Jumlah Dosen', fontsize=11)
ax.set_title('Top 10 Program Studi berdasarkan Jumlah Dosen', fontsize=13, fontweight='bold', pad=15)
ax.set_xlim(0, top_prodi.max() * 1.15)
plt.tight_layout()
plt.savefig('plot_01_dosen_per_prodi.png', bbox_inches='tight')
plt.show()

### 3.2 Tren Publikasi per Tahun

Lihat bagaimana jumlah publikasi dosen Infokom berkembang dari tahun ke tahun, dibedakan berdasarkan sumber (Scholar vs Scopus).

In [ ]:
trend = relasi.groupby(['Year', 'source_db']).size().unstack(fill_value=0)
trend = trend[trend.index.between(2010, 2025)]

print('=== TREN PUBLIKASI PER TAHUN ===')
print(trend.to_string())
print(f'\nTotal publikasi Scholar (2010-2025) : {trend["Scholar"].sum() if "Scholar" in trend else 0}')
print(f'Total publikasi Scopus  (2010-2025) : {trend["Scopus"].sum() if "Scopus" in trend else 0}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

colors_map = {'Scholar': PALETTE[0], 'Scopus': PALETTE[1]}
for col in trend.columns:
    axes[0].plot(trend.index, trend[col], marker='o', linewidth=2.5,
                 label=col, color=colors_map.get(col, PALETTE[2]))

axes[0].set_title('Tren Publikasi per Tahun per Sumber', fontweight='bold')
axes[0].set_xlabel('Tahun')
axes[0].set_ylabel('Jumlah Publikasi')
axes[0].legend()
axes[0].set_xticks(trend.index)
axes[0].tick_params(axis='x', rotation=45)

trend_total = trend.sum(axis=1)
axes[1].fill_between(trend_total.index, trend_total.values, alpha=0.4, color=PALETTE[2])
axes[1].plot(trend_total.index, trend_total.values, marker='o', linewidth=2.5, color=PALETTE[2])
axes[1].set_title('Total Publikasi per Tahun (Gabungan)', fontweight='bold')
axes[1].set_xlabel('Tahun')
axes[1].set_ylabel('Total Publikasi')
axes[1].set_xticks(trend_total.index)
axes[1].tick_params(axis='x', rotation=45)

plt.suptitle('Tren Produktivitas Riset Dosen Infokom', fontsize=14, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('plot_02_tren_publikasi.png', bbox_inches='tight')
plt.show()

### 3.3 Top 20 Dosen Paling Produktif

Hitung jumlah paper per dosen dari relasi yang sudah terbangun.

In [ ]:
produktivitas = relasi.groupby('nama_norm_clean').size().reset_index(name='total_paper')
produktivitas = produktivitas.merge(dosen[['nama_norm_clean', 'prodi']], on='nama_norm_clean', how='left')
produktivitas = produktivitas.sort_values('total_paper', ascending=False)

print('=== TOP 20 DOSEN PALING PRODUKTIF ===')
print(produktivitas.head(20).to_string(index=False))
print(f'\nRata-rata paper per dosen : {produktivitas["total_paper"].mean():.1f}')
print(f'Median paper per dosen    : {produktivitas["total_paper"].median():.1f}')
print(f'Maks paper (1 dosen)      : {produktivitas["total_paper"].max()}')

In [ ]:
top20 = produktivitas.head(20)

fig, ax = plt.subplots(figsize=(11, 7))

colors = [PALETTE[0] if i < 3 else PALETTE[1] if i < 10 else PALETTE[2] 
          for i in range(len(top20))]

bars = ax.barh(top20['nama_norm_clean'][::-1], top20['total_paper'][::-1],
               color=colors[::-1], edgecolor='white')

for bar, val in zip(bars, top20['total_paper'][::-1]):
    ax.text(bar.get_width() + 1, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9, fontweight='bold')

legend_patches = [
    mpatches.Patch(color=PALETTE[0], label='Top 3'),
    mpatches.Patch(color=PALETTE[1], label='Top 4-10'),
    mpatches.Patch(color=PALETTE[2], label='Top 11-20'),
]
ax.legend(handles=legend_patches, loc='lower right')
ax.set_xlabel('Jumlah Paper', fontsize=11)
ax.set_title('Top 20 Dosen Infokom Paling Produktif', fontsize=13, fontweight='bold', pad=15)
ax.set_xlim(0, top20['total_paper'].max() * 1.12)
plt.tight_layout()
plt.savefig('plot_03_top20_produktif.png', bbox_inches='tight')
plt.show()

### 3.4 Distribusi Produktivitas (Histogram)

Melihat apakah distribusi produktivitas merata atau terkonsentrasi pada sebagian kecil dosen (fenomena Power Law yang umum di riset akademik).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

axes[0].hist(produktivitas['total_paper'], bins=20, color=PALETTE[0], edgecolor='white', linewidth=0.7)
axes[0].axvline(produktivitas['total_paper'].mean(), color=PALETTE[3], linestyle='--', 
                linewidth=2, label=f'Rata-rata: {produktivitas["total_paper"].mean():.1f}')
axes[0].axvline(produktivitas['total_paper'].median(), color=PALETTE[4], linestyle='--',
                linewidth=2, label=f'Median: {produktivitas["total_paper"].median():.1f}')
axes[0].set_xlabel('Jumlah Paper per Dosen')
axes[0].set_ylabel('Frekuensi')
axes[0].set_title('Distribusi Produktivitas Dosen', fontweight='bold')
axes[0].legend()

by_source = relasi.groupby('source_db').size()
axes[1].pie(by_source.values, labels=by_source.index, autopct='%1.1f%%',
            colors=[PALETTE[0], PALETTE[1]], startangle=90,
            wedgeprops={'edgecolor': 'white', 'linewidth': 2})
axes[1].set_title('Proporsi Publikasi: Scholar vs Scopus', fontweight='bold')

plt.tight_layout()
plt.savefig('plot_04_distribusi_produktivitas.png', bbox_inches='tight')
plt.show()

pct_80 = (produktivitas['total_paper'].cumsum() / produktivitas['total_paper'].sum()).searchsorted(0.8) + 1
print(f'Hukum Pareto: {pct_80} dosen ({pct_80/len(produktivitas)*100:.1f}%) menghasilkan 80% dari total publikasi.')

### 3.5 Top Jurnal / Venue Publikasi

Jurnal atau konferensi apa yang paling sering digunakan dosen Infokom untuk publikasi?

In [ ]:
top_jurnal = (relasi.groupby(['Journal_clean', 'source_db'])
              .size()
              .reset_index(name='count')
              .sort_values('count', ascending=False)
              .head(15))

print('=== TOP 15 JURNAL/VENUE PUBLIKASI ===')
print(top_jurnal.to_string(index=False))

In [ ]:
top_jurnal_plot = top_jurnal.copy()
top_jurnal_plot['Journal_short'] = top_jurnal_plot['Journal_clean'].str[:50]

color_map = {'Scholar': PALETTE[0], 'Scopus': PALETTE[1]}
bar_colors = [color_map.get(src, PALETTE[2]) for src in top_jurnal_plot['source_db']]

fig, ax = plt.subplots(figsize=(11, 7))
bars = ax.barh(top_jurnal_plot['Journal_short'][::-1], top_jurnal_plot['count'][::-1],
               color=bar_colors[::-1], edgecolor='white')

for bar, val in zip(bars, top_jurnal_plot['count'][::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)

legend_patches = [mpatches.Patch(color=PALETTE[0], label='Scholar'),
                  mpatches.Patch(color=PALETTE[1], label='Scopus')]
ax.legend(handles=legend_patches)
ax.set_xlabel('Jumlah Paper')
ax.set_title('Top 15 Jurnal/Venue Publikasi Dosen Infokom', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('plot_05_top_jurnal.png', bbox_inches='tight')
plt.show()

### 3.6 Analisis Keyword / Topik Riset (Scopus)

Scopus memiliki data Keywords yang terstruktur. Kita analisis topik riset apa yang paling banyak diteliti.

In [ ]:
all_keywords = []
for kw_str in scopus_clean['Keywords'].dropna():
    keywords = [k.strip().lower() for k in str(kw_str).split(';') if k.strip()]
    all_keywords.extend(keywords)

kw_counter = Counter(all_keywords)
top_kw = pd.DataFrame(kw_counter.most_common(25), columns=['keyword', 'count'])

print('=== TOP 25 KEYWORD RISET (SCOPUS) ===')
print(top_kw.to_string(index=False))

In [ ]:
top20_kw = top_kw.head(20)

fig, ax = plt.subplots(figsize=(10, 7))
cmap_vals = plt.cm.Blues(np.linspace(0.4, 0.9, len(top20_kw)))
bars = ax.barh(top20_kw['keyword'][::-1], top20_kw['count'][::-1],
               color=cmap_vals, edgecolor='white')

for bar, val in zip(bars, top20_kw['count'][::-1]):
    ax.text(bar.get_width() + 0.2, bar.get_y() + bar.get_height()/2,
            str(val), va='center', fontsize=9)

ax.set_xlabel('Frekuensi')
ax.set_title('Top 20 Keyword Riset Dosen Infokom (Scopus)', fontsize=13, fontweight='bold', pad=15)
plt.tight_layout()
plt.savefig('plot_06_top_keywords.png', bbox_inches='tight')
plt.show()

---
## FASE 4 — Network Analysis

Bangun **graf kolaborasi** di mana:
- **Node** = Dosen
- **Edge** = pernah menulis paper bersama
- **Weight edge** = berapa kali berkolaborasi

Lalu hitung metrik jaringan: degree, betweenness centrality, clustering coefficient, dan deteksi komunitas.

### 4.1 Bangun Graf Kolaborasi

In [ ]:
paper_authors = relasi.groupby('paper_id')['nama_norm_clean'].apply(list).reset_index()
paper_authors = paper_authors[paper_authors['nama_norm_clean'].map(len) >= 2]

edge_counter = Counter()
for _, row in paper_authors.iterrows():
    authors = sorted(set(row['nama_norm_clean']))
    for pair in combinations(authors, 2):
        edge_counter[pair] += 1

G = nx.Graph()

for dosen_name in relasi['nama_norm_clean'].unique():
    prodi_val = dosen[dosen['nama_norm_clean'] == dosen_name]['prodi'].values
    prodi_val = prodi_val[0] if len(prodi_val) > 0 else 'Unknown'
    G.add_node(dosen_name, prodi=prodi_val)

for (u, v), w in edge_counter.items():
    G.add_edge(u, v, weight=w)

print('=== STATISTIK GRAF KOLABORASI ===')
print(f'Jumlah node (dosen)   : {G.number_of_nodes()}')
print(f'Jumlah edge (kolaborasi) : {G.number_of_edges()}')
print(f'Density graf          : {nx.density(G):.4f}')
print(f'Apakah connected      : {nx.is_connected(G)}')
components = list(nx.connected_components(G))
print(f'Jumlah komponen       : {len(components)}')
print(f'Komponen terbesar     : {max(len(c) for c in components)} node')

### 4.2 Hitung Metrik Sentralitas

Empat metrik kunci:
- **Degree Centrality**: seberapa banyak koneksi langsung
- **Betweenness Centrality**: seberapa sering jadi "jembatan" antar dosen
- **Closeness Centrality**: seberapa dekat ke semua node lain
- **Clustering Coefficient**: seberapa erat kelompok kolaborasinya

In [ ]:
Gcc = G.subgraph(max(nx.connected_components(G), key=len)).copy()

degree_cent     = nx.degree_centrality(Gcc)
betweenness     = nx.betweenness_centrality(Gcc, weight='weight', normalized=True)
closeness       = nx.closeness_centrality(Gcc)
clustering      = nx.clustering(Gcc, weight='weight')
degree_raw      = dict(Gcc.degree(weight='weight'))

metrics_df = pd.DataFrame({
    'dosen': list(Gcc.nodes()),
    'weighted_degree': [degree_raw[n] for n in Gcc.nodes()],
    'degree_centrality': [degree_cent[n] for n in Gcc.nodes()],
    'betweenness': [betweenness[n] for n in Gcc.nodes()],
    'closeness': [closeness[n] for n in Gcc.nodes()],
    'clustering': [clustering[n] for n in Gcc.nodes()],
})

metrics_df = metrics_df.merge(
    dosen[['nama_norm_clean', 'prodi']], 
    left_on='dosen', right_on='nama_norm_clean', how='left'
).drop(columns='nama_norm_clean')

metrics_df = metrics_df.sort_values('betweenness', ascending=False)

print('=== TOP 15 DOSEN BERDASARKAN BETWEENNESS CENTRALITY ===')
print('(Betweenness tinggi = berperan sebagai jembatan kolaborasi antar kelompok)')
print()
print(metrics_df.head(15)[['dosen','weighted_degree','betweenness','closeness','clustering']]
      .to_string(index=False))

Ringkasan statistik semua metrik sentralitas.

In [ ]:
print('=== STATISTIK DESKRIPTIF METRIK JARINGAN ===')
print(metrics_df[['weighted_degree','degree_centrality','betweenness','closeness','clustering']]
      .describe().to_string())

### 4.3 Visualisasi Metrik Sentralitas (Scatter Plot)

Plot hubungan antara Degree (banyak kolaborasi) vs Betweenness (jadi jembatan). Dosen ideal berada di kanan atas.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sc = axes[0].scatter(
    metrics_df['degree_centrality'],
    metrics_df['betweenness'],
    c=metrics_df['clustering'],
    cmap='coolwarm',
    s=metrics_df['weighted_degree'] * 3 + 30,
    alpha=0.75,
    edgecolors='white',
    linewidths=0.5
)
plt.colorbar(sc, ax=axes[0], label='Clustering Coefficient')

top5_between = metrics_df.nlargest(5, 'betweenness')
for _, row in top5_between.iterrows():
    axes[0].annotate(row['dosen'], (row['degree_centrality'], row['betweenness']),
                     fontsize=7, xytext=(5, 3), textcoords='offset points')

axes[0].set_xlabel('Degree Centrality')
axes[0].set_ylabel('Betweenness Centrality')
axes[0].set_title('Degree vs Betweenness Centrality\n(ukuran titik = weighted degree)', fontweight='bold')

top15_degree = metrics_df.nlargest(15, 'weighted_degree')
axes[1].barh(top15_degree['dosen'][::-1], top15_degree['weighted_degree'][::-1],
             color=PALETTE[1], edgecolor='white')
axes[1].set_xlabel('Weighted Degree (total kolaborasi)')
axes[1].set_title('Top 15 Dosen: Weighted Degree\n(total bobot semua kolaborasi)', fontweight='bold')

plt.suptitle('Analisis Sentralitas Jaringan Kolaborasi', fontsize=13, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('plot_07_sentralitas.png', bbox_inches='tight')
plt.show()

### 4.4 Deteksi Komunitas (Louvain / Greedy Modularity)

Algoritma deteksi komunitas mengelompokkan dosen yang cenderung saling berkolaborasi satu sama lain.

In [ ]:
communities = list(nx.community.greedy_modularity_communities(Gcc, weight='weight'))
communities_sorted = sorted(communities, key=len, reverse=True)

node_community = {}
for i, comm in enumerate(communities_sorted):
    for node in comm:
        node_community[node] = i

nx.set_node_attributes(Gcc, node_community, 'community')

print(f'Jumlah komunitas terdeteksi : {len(communities_sorted)}')
print()
for i, comm in enumerate(communities_sorted[:8]):
    print(f'Komunitas {i+1} ({len(comm)} dosen): {sorted(list(comm))[:6]}{" ..." if len(comm) > 6 else ""}')

### 4.5 Visualisasi Graf Kolaborasi

Visualisasi graf dengan warna node berdasarkan komunitas, ukuran node berdasarkan betweenness centrality, dan ketebalan edge berdasarkan jumlah kolaborasi.

In [ ]:
min_collab = 2
G_filtered = nx.Graph()

for n, data in Gcc.nodes(data=True):
    G_filtered.add_node(n, **data)

for u, v, d in Gcc.edges(data=True):
    if d.get('weight', 1) >= min_collab:
        G_filtered.add_edge(u, v, **d)

G_filtered.remove_nodes_from(list(nx.isolates(G_filtered)))

print(f'Graf setelah filter (min {min_collab} kolaborasi):')
print(f'  Node : {G_filtered.number_of_nodes()}')
print(f'  Edge : {G_filtered.number_of_edges()}')

In [ ]:
pos = nx.spring_layout(G_filtered, seed=42, k=2.5)

community_colors = plt.cm.tab20.colors
node_colors = [community_colors[node_community.get(n, 0) % len(community_colors)] 
               for n in G_filtered.nodes()]

node_sizes = [betweenness.get(n, 0.001) * 8000 + 100 for n in G_filtered.nodes()]
edge_weights = [G_filtered[u][v].get('weight', 1) for u, v in G_filtered.edges()]
edge_widths  = [0.5 + w * 0.4 for w in edge_weights]

fig, ax = plt.subplots(figsize=(16, 12))
ax.set_facecolor('#F8F9FA')

nx.draw_networkx_edges(G_filtered, pos, ax=ax,
                       width=edge_widths,
                       edge_color='#AAAAAA',
                       alpha=0.5)

nx.draw_networkx_nodes(G_filtered, pos, ax=ax,
                       node_color=node_colors,
                       node_size=node_sizes,
                       alpha=0.9,
                       linewidths=1,
                       edgecolors='white')

top_nodes = sorted(betweenness.keys(), key=lambda n: betweenness[n], reverse=True)[:15]
top_nodes_in_graph = [n for n in top_nodes if n in G_filtered.nodes()]
labels = {n: n for n in top_nodes_in_graph}

nx.draw_networkx_labels(G_filtered, pos, labels=labels, ax=ax,
                        font_size=7, font_weight='bold', font_color='#1a1a1a')

num_communities = len(set(node_community.get(n, 0) for n in G_filtered.nodes()))
patches = [mpatches.Patch(color=community_colors[i % len(community_colors)], 
                           label=f'Komunitas {i+1}')
           for i in range(min(num_communities, 8))]
ax.legend(handles=patches, loc='lower left', fontsize=8, 
          title='Kelompok Kolaborasi', title_fontsize=9)

ax.set_title('Graf Kolaborasi Dosen Infokom\n(Ukuran node = Betweenness Centrality | Warna = Komunitas | Tebal edge = Frekuensi kolaborasi)',
             fontsize=13, fontweight='bold', pad=20)
ax.axis('off')
plt.tight_layout()
plt.savefig('plot_08_graf_kolaborasi.png', bbox_inches='tight', dpi=150)
plt.show()

### 4.6 Analisis Kolaborasi Antar Prodi

Hitung seberapa banyak paper yang melibatkan dosen dari prodi berbeda (kolaborasi lintas prodi).

In [ ]:
paper_prodi = relasi.groupby('paper_id')['prodi'].apply(list)

cross_prodi_edges = Counter()
for paper_id, prodi_list in paper_prodi.items():
    prodi_unique = sorted(set(p for p in prodi_list if pd.notna(p)))
    for pair in combinations(prodi_unique, 2):
        cross_prodi_edges[pair] += 1

cross_df = pd.DataFrame([
    {'prodi_a': k[0], 'prodi_b': k[1], 'kolaborasi': v}
    for k, v in cross_prodi_edges.most_common(15)
])

cross_single = relasi.groupby('paper_id')['prodi'].nunique()
lintas_prodi = (cross_single > 1).sum()
satu_prodi   = (cross_single == 1).sum()

print(f'Paper dengan kolaborasi LINTAS prodi  : {lintas_prodi} ({lintas_prodi/len(cross_single)*100:.1f}%)')
print(f'Paper dari satu prodi saja            : {satu_prodi} ({satu_prodi/len(cross_single)*100:.1f}%)')
print()
print('=== TOP 15 PASANGAN PRODI YANG PALING BANYAK BERKOLABORASI ===')
print(cross_df.to_string(index=False))

---
## FASE 5 — Import ke Neo4j

Setelah analisis selesai, kita muat data ke Neo4j sebagai Knowledge Graph. Struktur yang dibuat:
- **Node**: `(:Dosen)`, `(:Paper)`, `(:Jurnal)`, `(:Prodi)`
- **Relationship**: `(:Dosen)-[:MENULIS]->(:Paper)`, `(:Paper)-[:DITERBITKAN_DI]->(:Jurnal)`, `(:Dosen)-[:MENGAJAR_DI]->(:Prodi)`

Ganti `NEO4J_PASSWORD` dengan password yang kamu buat saat membuat instance.

In [ ]:
from neo4j import GraphDatabase

NEO4J_URI      = 'neo4j://127.0.0.1:7687'
NEO4J_USER     = 'neo4j'
NEO4J_PASSWORD = 'ISI_PASSWORD_KAMU_DISINI'

driver = GraphDatabase.driver(NEO4J_URI, auth=(NEO4J_USER, NEO4J_PASSWORD))

try:
    driver.verify_connectivity()
    print('Koneksi ke Neo4j berhasil!')
except Exception as e:
    print(f'Koneksi GAGAL: {e}')

### 5.1 Bersihkan Database & Buat Constraint

Hapus semua data lama (jika ada) dan buat unique constraint agar tidak ada node duplikat.

In [ ]:
with driver.session() as session:
    session.run('MATCH (n) DETACH DELETE n')
    print('Database dikosongkan.')

    constraints = [
        'CREATE CONSTRAINT IF NOT EXISTS FOR (d:Dosen) REQUIRE d.nama IS UNIQUE',
        'CREATE CONSTRAINT IF NOT EXISTS FOR (p:Paper) REQUIRE p.paper_id IS UNIQUE',
        'CREATE CONSTRAINT IF NOT EXISTS FOR (j:Jurnal) REQUIRE j.nama IS UNIQUE',
        'CREATE CONSTRAINT IF NOT EXISTS FOR (pr:Prodi) REQUIRE pr.nama IS UNIQUE',
    ]
    for c in constraints:
        session.run(c)
    print('Constraint berhasil dibuat.')

### 5.2 Import Node Dosen & Prodi

In [ ]:
def batch_run(session, query, data, batch_size=200):
    total = 0
    for i in range(0, len(data), batch_size):
        batch = data[i:i+batch_size]
        session.run(query, rows=batch)
        total += len(batch)
    return total

dosen_records = [
    {
        'nama': row['nama_norm_clean'],
        'nama_lengkap': row['nama_dosen'],
        'prodi': row['prodi'],
        'scholar_id': str(row['scholar_id']) if pd.notna(row['scholar_id']) else '',
        'scopus_id': str(row['scopus_id']) if pd.notna(row['scopus_id']) else '',
        'nidn': str(row['nidn']) if pd.notna(row['nidn']) else '',
    }
    for _, row in dosen.iterrows()
]

query_dosen = '''
UNWIND $rows AS row
MERGE (d:Dosen {nama: row.nama})
  SET d.nama_lengkap = row.nama_lengkap,
      d.scholar_id   = row.scholar_id,
      d.scopus_id    = row.scopus_id,
      d.nidn         = row.nidn
MERGE (pr:Prodi {nama: row.prodi})
MERGE (d)-[:MENGAJAR_DI]->(pr)
'''

with driver.session() as session:
    n = batch_run(session, query_dosen, dosen_records)
    print(f'Berhasil import {n} dosen beserta relasi ke Prodi.')

### 5.3 Import Node Paper & Relasi MENULIS

In [ ]:
paper_meta = relasi.drop_duplicates(subset='paper_id')[['paper_id','Title','Year','Journal_clean','source_db']]

paper_records = [
    {
        'paper_id': str(row['paper_id']),
        'title': str(row['Title'])[:300] if pd.notna(row['Title']) else '',
        'year': int(row['Year']) if pd.notna(row['Year']) else 0,
        'journal': str(row['Journal_clean']) if pd.notna(row['Journal_clean']) else 'Unknown',
        'source': str(row['source_db']),
    }
    for _, row in paper_meta.iterrows()
]

query_paper = '''
UNWIND $rows AS row
MERGE (p:Paper {paper_id: row.paper_id})
  SET p.title  = row.title,
      p.year   = row.year,
      p.source = row.source
MERGE (j:Jurnal {nama: row.journal})
MERGE (p)-[:DITERBITKAN_DI]->(j)
'''

with driver.session() as session:
    n = batch_run(session, query_paper, paper_records)
    print(f'Berhasil import {n} paper beserta relasi ke Jurnal.')

In [ ]:
relasi_records = [
    {
        'dosen': row['nama_norm_clean'],
        'paper_id': str(row['paper_id']),
    }
    for _, row in relasi.iterrows()
    if pd.notna(row['nama_norm_clean']) and row['nama_norm_clean'] != ''
]

query_menulis = '''
UNWIND $rows AS row
MATCH (d:Dosen {nama: row.dosen})
MATCH (p:Paper {paper_id: row.paper_id})
MERGE (d)-[:MENULIS]->(p)
'''

with driver.session() as session:
    n = batch_run(session, query_menulis, relasi_records)
    print(f'Berhasil import {n} relasi MENULIS.')

### 5.4 Import Relasi BERKOLABORASI Antar Dosen

In [ ]:
collab_records = [
    {'dosen_a': pair[0], 'dosen_b': pair[1], 'weight': count}
    for pair, count in edge_counter.items()
]

query_collab = '''
UNWIND $rows AS row
MATCH (a:Dosen {nama: row.dosen_a})
MATCH (b:Dosen {nama: row.dosen_b})
MERGE (a)-[r:BERKOLABORASI]-(b)
  SET r.weight = row.weight
'''

with driver.session() as session:
    n = batch_run(session, query_collab, collab_records)
    print(f'Berhasil import {n} relasi BERKOLABORASI.')

### 5.5 Verifikasi Import & Query Sampel

Jalankan beberapa query Cypher untuk memverifikasi data sudah masuk dengan benar.

In [ ]:
queries = {
    'Jumlah Dosen'  : 'MATCH (d:Dosen) RETURN count(d) AS total',
    'Jumlah Paper'  : 'MATCH (p:Paper) RETURN count(p) AS total',
    'Jumlah Jurnal' : 'MATCH (j:Jurnal) RETURN count(j) AS total',
    'Jumlah Prodi'  : 'MATCH (pr:Prodi) RETURN count(pr) AS total',
    'Relasi MENULIS': 'MATCH ()-[r:MENULIS]->() RETURN count(r) AS total',
    'Relasi KOLABORASI': 'MATCH ()-[r:BERKOLABORASI]-() RETURN count(r)/2 AS total',
}

print('=== VERIFIKASI ISI GRAPH DATABASE ===')
with driver.session() as session:
    for label, q in queries.items():
        result = session.run(q).single()
        print(f'  {label:<22} : {result["total"]}')

In [ ]:
print('=== TOP 10 DOSEN PALING PRODUKTIF (dari Neo4j) ===')
query_top_produktif = '''
MATCH (d:Dosen)-[:MENULIS]->(p:Paper)
RETURN d.nama AS dosen, count(p) AS total_paper
ORDER BY total_paper DESC
LIMIT 10
'''
with driver.session() as session:
    results = session.run(query_top_produktif)
    for i, row in enumerate(results, 1):
        print(f'  {i:2}. {row["dosen"]:<30} : {row["total_paper"]} paper')

In [ ]:
print('=== TOP 10 DOSEN PALING BANYAK BERKOLABORASI (dari Neo4j) ===')
query_top_collab = '''
MATCH (d:Dosen)-[r:BERKOLABORASI]-(other:Dosen)
RETURN d.nama AS dosen, count(DISTINCT other) AS mitra, sum(r.weight) AS total_kolaborasi
ORDER BY total_kolaborasi DESC
LIMIT 10
'''
with driver.session() as session:
    results = session.run(query_top_collab)
    for i, row in enumerate(results, 1):
        print(f'  {i:2}. {row["dosen"]:<30} | Mitra: {row["mitra"]:3} | Total kolaborasi: {row["total_kolaborasi"]}')

In [ ]:
driver.close()
print('Koneksi Neo4j ditutup.')
print()
print('=== SELESAI ===')
print('Semua fase berhasil dijalankan:')
print('  [1] Data Loading & Audit')
print('  [2] Preprocessing & Normalisasi')
print('  [3] Exploratory Data Analysis')
print('  [4] Network Analysis (Graf, Sentralitas, Komunitas)')
print('  [5] Import ke Neo4j')

---
## Ringkasan Hasil Analisis

Setelah seluruh notebook dijalankan, kamu memiliki:

**Dari EDA:**
- Distribusi dosen per prodi dan tren produktivitas per tahun
- Ranking dosen paling produktif berdasarkan jumlah publikasi
- Jurnal/venue yang paling sering digunakan
- Topik riset dominan berdasarkan keyword Scopus

**Dari Network Analysis:**
- Graf kolaborasi dengan metrik Degree, Betweenness, Closeness, Clustering
- Identifikasi dosen yang berperan sebagai jembatan kolaborasi (Betweenness tinggi)
- Kelompok/komunitas riset yang terbentuk secara alami
- Pola kolaborasi lintas prodi

**Di Neo4j (bisa diquery langsung):**
```cypher
// Siapa yang paling banyak berkolaborasi dengan satu dosen tertentu?
MATCH (d:Dosen {nama: 'Yuni Yamasari'})-[r:BERKOLABORASI]-(other)
RETURN other.nama, r.weight ORDER BY r.weight DESC LIMIT 10

// Paper apa yang melibatkan paling banyak dosen Infokom?
MATCH (d:Dosen)-[:MENULIS]->(p:Paper)
RETURN p.title, count(d) AS penulis ORDER BY penulis DESC LIMIT 5
```